# TRANSFORMERS+SUBSETS





In [ ]:
!pip install -U transformers accelerate #Installa e aggiorna le librerie fondamentali per caricare i modelli Transformer e ottimizzare l'esecuzione del codice su diverse configurazioni hardware (come le GPU).

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 70.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from datasets import load_dataset
import pandas as pd

# 1. Carica il dataset
dataset = load_dataset("squad", split="validation")

# 2. Lista dei titoli da estrarre
targets = ['Apollo_program', 'Yuan_dynasty', 'Teacher']

for title in targets:
    # Filtra ed esporta
    filtered_ds = dataset.filter(lambda x: x['title'] == title)
    df = pd.DataFrame(filtered_ds)
    df.to_csv(f"{title.lower()}_questions.csv", index=False)
    print(f"Estratte {len(df)} domande su {title}!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

...

Estratte 242 domande su Apollo_program!


Filter:   0%|          | 0/10570 [00:00<?, ? examples/s]

Estratte 228 domande su Yuan_dynasty!


Filter:   0%|          | 0/10570 [00:00<?, ? examples/s]

Estratte 268 domande su Teacher!


-----
#GEMMA
-----

In [ ]:
from tqdm import tqdm
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# 0. Login Hugging Face
login(token=os.environ.get("HF_TOKEN"))
# 1. Usa Gemma-3-4B-IT
model_name = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer.pad_token = tokenizer.eos_token

#
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,  # Mantiene efficienza memoria
)

# Sposta manualmente il modello su GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"✅ Modello caricato su {device}")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✅ Modello caricato su cuda


------
#QWEN
------

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Login Hugging Face
login(token=os.environ.get("HF_TOKEN"))

# --- CONFIGURAZIONE MODELLO ---
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
# Carica tokenizer e modello
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

#ZERO-SHOT          


In [ ]:
import pandas as pd
import torch
import os

# --- CONFIGURAZIONE ESPERIMENTO ---
MODEL_NAME = "google/gemma-3-4b-it"  # Qwen/Qwen3-4B-Instruct-2507 | google/gemma-3-4b-it
METHOD_NAME = "Zero-Shot"
NUM_RUNS = 1 # Quante NUOVE run vuoi fare (es. se metti 1 e l'ultima era la 3, farà solo la 4)
filenames = ["apollo_program_questions.csv", "teacher_questions.csv", "yuan_dynasty_questions.csv"]

OUTPUT_FILE = "risultati_zero_shot.csv"
model_short = MODEL_NAME.split("/")[-1]  # es. "gemma-3-4b-it" o "Qwen3-4B-Instruct-2507"

# --- FUNZIONE DI QUERY ---
def zero_shot_query(question):
    content = f"Provide a brief answer (maximum 10 words).\n\nQuestion: {question}\nAnswer:"

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
            temperature=0.1
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# --- CALCOLO RUN INIZIALE ---
start_run = 1

if os.path.exists(OUTPUT_FILE):
    try:
        existing_df = pd.read_csv(OUTPUT_FILE, usecols=['model_name', 'run_id'])
        if not existing_df.empty:
            model_df = existing_df[existing_df['model_name'] == MODEL_NAME]
            if not model_df.empty:
                last_run = model_df['run_id'].max()
                start_run = last_run + 1
                print(f"📂 File trovato! Per '{model_short}' l'ultima run è la {last_run}. Inizierò dalla Run {start_run}.")
            else:
                print(f"📂 File trovato, ma nessuna run precedente per '{model_short}'. Inizio dalla Run 1.")
        else:
            print("📂 File trovato ma vuoto. Inizio dalla Run 1.")
    except Exception as e:
        print(f"⚠️ Errore nella lettura del file esistente ({e}). Inizio dalla Run 1.")
else:
    print("🆕 Nessun file precedente trovato. Inizio dalla Run 1.")

# --- ESECUZIONE RUN CON SALVATAGGIO INCREMENTALE ---
for run_id in range(start_run, start_run + NUM_RUNS):
    print(f"\n🚀 AVVIO RUN {run_id} di {model_short}")

    for file in filenames:
        domain = file.replace("_questions.csv", "")
        print(f"--- Dominio: {domain} ---")

        try:
            df = pd.read_csv(file)
        except FileNotFoundError:
            print(f"❌ Errore: Il file {file} non è stato trovato.")
            continue

        for index, row in df.iterrows():
            domanda = row['question']
            ground_truth = row['answers']

            risposta_modello = zero_shot_query(domanda)

            new_record = pd.DataFrame([{
                "model_name": MODEL_NAME,
                "method": METHOD_NAME,
                "domain": domain,
                "run_id": run_id,
                "question": domanda,
                "ground_truth": ground_truth,
                "llm_answer": risposta_modello
            }])

            if os.path.exists(OUTPUT_FILE):
                new_record.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8')
            else:
                new_record.to_csv(OUTPUT_FILE, mode='w', header=True, index=False, encoding='utf-8')

            if index % 200 == 0:
                print(f"✅ Salvata risposta {index+1}/{len(df)} (Run {run_id}, {domain})")

print(f"\n🎉 Esperimento completato per '{model_short}'! Run eseguite: da {start_run} a {start_run + NUM_RUNS - 1}")

📂 File trovato! Per 'gemma-3-4b-it' l'ultima run è la 29. Inizierò dalla Run 30.

🚀 AVVIO RUN 30 di gemma-3-4b-it
--- Dominio: apollo_program ---
✅ Salvata risposta 1/242 (Run 30, apollo_program)
✅ Salvata risposta 201/242 (Run 30, apollo_program)
--- Dominio: teacher ---
✅ Salvata risposta 1/268 (Run 30, teacher)
✅ Salvata risposta 201/268 (Run 30, teacher)
--- Dominio: yuan_dynasty ---
✅ Salvata risposta 1/228 (Run 30, yuan_dynasty)
✅ Salvata risposta 201/228 (Run 30, yuan_dynasty)

🚀 AVVIO RUN 31 di gemma-3-4b-it
--- Dominio: apollo_program ---
✅ Salvata risposta 1/242 (Run 31, apollo_program)
✅ Salvata risposta 201/242 (Run 31, apollo_program)
--- Dominio: teacher ---
✅ Salvata risposta 1/268 (Run 31, teacher)
✅ Salvata risposta 201/268 (Run 31, teacher)
--- Dominio: yuan_dynasty ---
✅ Salvata risposta 1/228 (Run 31, yuan_dynasty)
✅ Salvata risposta 201/228 (Run 31, yuan_dynasty)

🚀 AVVIO RUN 32 di gemma-3-4b-it
--- Dominio: apollo_program ---
✅ Salvata risposta 1/242 (Run 32, apol


#**IN-CONTEXT LEARNING**



In [ ]:
import pandas as pd
import torch
import os

# --- CONFIGURAZIONE ESPERIMENTO ---
MODEL_NAME = "google/gemma-3-4b-it"   #Qwen/Qwen3-4B-Instruct-2507 | google/gemma-3-4b-it
METHOD_NAME = "In-Context-Learning"
NUM_RUNS = 1
filenames = ["apollo_program_questions.csv", "teacher_questions.csv", "yuan_dynasty_questions.csv"]

# Nome del file di output
OUTPUT_FILE = "risultati_in_context_learning.csv"

# Funzione per l'interrogazione con Contesto (In-Context)
def in_context_learning_query(question, context):
    content = (
        f"Use the provided context to answer the question. Don't answer with \"According to the text\" \n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{question}"
    )

    messages = [{"role": "user", "content": content}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
            temperature=0.1  # Temperatura bassa per stabilità
        )

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# --- ESECUZIONE RUN CON SALVATAGGIO INCREMENTALE ---
for run_id in range(1, NUM_RUNS + 1):
    print(f"\n🚀 AVVIO RUN {run_id} di {NUM_RUNS}")

    for file in filenames:
        domain = file.replace("_questions.csv", "")  # Estrae il nome del dominio dal file
        print(f"--- Dominio: {domain} ---")

        df = pd.read_csv(file)

        for index, row in df.iterrows():
            domanda = row['question']
            contesto = row['context']
            ground_truth = row['answers']

            # Genera la risposta
            risposta_modello = in_context_learning_query(domanda, contesto)

            # --- STAMPA DI DEBUG ---
            print(f"\n[Domanda {index + 1}/{len(df)}] {domanda}")
            print(f"💡 Risposta LLM: {risposta_modello}")
            print("-" * 60)

            # Crea il nuovo record
            new_record = pd.DataFrame([{
                "model_name": MODEL_NAME,
                "method": METHOD_NAME,
                "domain": domain,
                "run_id": run_id,
                "question": domanda,
                "context": contesto,
                "ground_truth": ground_truth,
                "llm_answer": risposta_modello
            }])

            # --- SALVATAGGIO INCREMENTALE ---
            if os.path.exists(OUTPUT_FILE):
                # Appendi al file esistente senza rileggere tutto
                new_record.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8')
            else:
                # Crea il file con intestazione
                new_record.to_csv(OUTPUT_FILE, mode='w', header=True, index=False, encoding='utf-8')

            print(f"💾 Salvato: {domain} - Domanda {index + 1}")

# --- MESSAGGIO FINALE ---
df_final = pd.read_csv(OUTPUT_FILE)
print(f"\n✅ Esperimento completato!")
print(f"📊 File '{OUTPUT_FILE}' contiene {len(df_final)} righe totali.")


🚀 AVVIO RUN 1 di 1
--- Dominio: apollo_program ---

[Domanda 1/242] What project put the first Americans into space?
💡 Risposta LLM: Project Mercury.
------------------------------------------------------------
💾 Salvato: apollo_program - Domanda 1

[Domanda 2/242] What program was created to carry out these projects and missions?
💡 Risposta LLM: The National Aeronautics and Space Administration (NASA).
------------------------------------------------------------
💾 Salvato: apollo_program - Domanda 2

[Domanda 3/242] What year did the first manned Apollo flight occur?
💡 Risposta LLM: 1968
------------------------------------------------------------
💾 Salvato: apollo_program - Domanda 3

[Domanda 4/242] What President is credited with the original notion of putting Americans in space?
💡 Risposta LLM: Dwight D. Eisenhower.
------------------------------------------------------------
💾 Salvato: apollo_program - Domanda 4

[Domanda 5/242] How many people were on the project that followed 

#agent WEB SEARCH

In [ ]:
!pip install -U transformers accelerate duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 65.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import time
import os
import warnings
import logging
import random

warnings.filterwarnings("ignore", category=DeprecationWarning)

from transformers import AutoModelForCausalLM, AutoTokenizer
from duckduckgo_search import DDGS
from huggingface_hub import login


login(token=os.environ.get("HF_TOKEN"))

MODEL_NAME = "google/gemma-3-4b-it" #Qwen/Qwen3-4B-Instruct-2507 | google/gemma-3-4b-it
METHOD_NAME = "Web-Search-Agent"
NUM_RUNS = 1
OUTPUT_FILENAME = "risultati_web_agent.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Loading {MODEL_NAME}...")


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})

print(f"   pad_token       : {repr(tokenizer.pad_token)}  (id={tokenizer.pad_token_id})")
print(f"   eos_token       : {repr(tokenizer.eos_token)}  (id={tokenizer.eos_token_id})")



model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda",          # lascia accelerate gestire il placement
    dtype=torch.bfloat16,       # "torch_dtype" è deprecato, usa "dtype"
    low_cpu_mem_usage=True,
)


model.config.pad_token_id = tokenizer.pad_token_id


model.resize_token_embeddings(len(tokenizer))

model.eval()
print(f"✅ Model loaded on {device}\n")


# ─────────────────────────────────────────────────────────────────────────────
# SEARCH FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def search_web(query, max_results=5, max_retries=3):
    backends = ["html", "lite"]

    for attempt in range(max_retries):
        for backend in backends:
            try:
                with DDGS() as ddgs:
                    results = list(
                        ddgs.text(
                            query,
                            max_results=max_results,
                            backend=backend,
                            region="us-en",
                            safesearch="moderate",
                        )
                    )
                    if results:
                        context_text = ""
                        for i, res in enumerate(results):
                            context_text += f"Source {i+1}: {res['title']}\nContent: {res['body']}\n\n"
                        return context_text, "duckduckgo"
            except Exception:
                continue

        if attempt < max_retries - 1:
            wait_time = random.uniform(3, 6) * (attempt + 1)
            print(f"    ⏳ Retry {attempt + 1}/{max_retries}, waiting {wait_time:.1f}s...")
            time.sleep(wait_time)

    return "No results found after multiple attempts.", "failed"


DOMAIN_KEYWORDS = {
    "apollo program": "NASA Apollo mission moon landing 1969",
    "yuan_dynasty": "Mongol Yuan dynasty China Kublai Khan 1271-1368",
    "teacher": "teacher education classroom instructor pedagogy",
}


def clean_query(user_question):
    stop_words = [
        "what", "when", "where", "who", "how", "is", "the", "a", "an",
        "tell", "me", "about", "did", "was", "were",
    ]
    words = user_question.replace("?", "").split()
    keywords = [w for w in words if w.lower() not in stop_words]
    return " ".join(keywords)


# ─────────────────────────────────────────────────────────────────────────────
# AGENT
# ─────────────────────────────────────────────────────────────────────────────
def agent_web_search_query(user_question, domain_topic):
    topic_clean = domain_topic.lower().strip()
    extra_context = DOMAIN_KEYWORDS.get(topic_clean, domain_topic)

    search_query = f"{user_question} {domain_topic}"
    print(f"   🔎 Searching: {search_query}...")

    web_text, status = search_web(search_query, max_results=5)

    if status == "failed" or "No results" in web_text:
        print("   ⚠️  Full query failed. Retrying with keywords...")
        keywords = clean_query(user_question)
        fallback_query = f"{extra_context} {keywords}"
        web_text, status = search_web(fallback_query, max_results=5)

    if status != "failed" and "No results" not in web_text:
        preview = web_text[:100].replace("\n", " ")
        print(f"   📄 Context found: {preview}...")
        final_status = "success"
    else:
        print("   ⚠️ No web results found. Using internal knowledge.")
        web_text = "No specific search results found on the web."
        final_status = "failed"

    system_instruction = "Provide a brief answer (maximum 10 words)."
    user_data = f"""Reference Context:
{web_text}

Question: {user_question}

Instruction: Use the context above if it helps answer the question. \
If the context is irrelevant or does not contain the answer, ignore it \
and answer based on your own knowledge. Do not mention if the context was used or not."""

    combined_prompt = f"{system_instruction}\n\n{user_data}"
    messages = [{"role": "user", "content": combined_prompt}]

    text_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


    model_inputs = tokenizer(
        [text_input],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(model.device)

    input_len = model_inputs["input_ids"].shape[1]


    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=model_inputs["input_ids"],
            attention_mask=model_inputs["attention_mask"],
            max_new_tokens=50,
            do_sample=True,
            temperature=0.1,
            pad_token_id=tokenizer.pad_token_id,   # ← FIX critico
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids_trimmed = generated_ids[:, input_len:]
    answer_only = tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

    print(f"   🤖 Answer: {answer_only[:60]}...")
    return answer_only.strip(), web_text, final_status


# ─────────────────────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────────────────────
filenames = ["apollo_program_questions.csv", "teacher_questions.csv", "yuan_dynasty_questions.csv"]
#filenames = ["apollo_program_questions.csv"]
#filenames = ["teacher_questions.csv"]
#filenames = ["yuan_dynasty_questions.csv"]

all_results = []

for run_id in range(1, NUM_RUNS + 1):
    print(f"\n{'='*70}")
    print(f"🚀 RUN {run_id}/{NUM_RUNS}")
    print(f"{'='*70}")

    for file in filenames:
        domain_raw = file.replace("_questions.csv", "")
        search_topic = domain_raw.replace("_", " ")

        print(f"\n📁 {domain_raw}")

        try:
            df = pd.read_csv(file)


            for idx, row in df.iterrows():
                question = row["question"]
                ground_truth = row["answers"]

                llm_answer, retrieved_context, method = agent_web_search_query(
                    question, search_topic
                )

                new_record = pd.DataFrame([{
                    "model_name": MODEL_NAME,
                    "method": METHOD_NAME,
                    "domain": domain_raw,
                    "run_id": run_id,
                    "question": question,
                    "web_search_query_used": f"{question} {search_topic}",
                    "retrieval_method": method,
                    "context": retrieved_context,
                    "ground_truth": ground_truth,
                    "llm_answer": llm_answer,
                }])

                all_results.append(new_record.iloc[0].to_dict())

                # ───  salvataggio incrementale  ─────────────────

                if os.path.exists(OUTPUT_FILENAME):
                    new_record.to_csv(OUTPUT_FILENAME, mode="a", header=False, index=False, encoding="utf-8")
                else:
                    new_record.to_csv(OUTPUT_FILENAME, mode="w", header=True, index=False, encoding="utf-8")

                print(f"   ✅ Saved {idx+1}/{len(df)} (Run {run_id}, {domain_raw})")

                sleep_time = random.uniform(4, 7) if method == "duckduckgo" else 1
                time.sleep(sleep_time)

        except Exception as e:
            print(f"   ❌ Errore durante l'elaborazione del file {file}: {e}")

print(f"\n🎉 Sperimentazione completata! I risultati sono in: {OUTPUT_FILENAME}")
print(f"{'='*70}\n")

🔧 Loading google/gemma-3-4b-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   pad_token       : '<pad>'  (id=0)
   eos_token       : '<eos>'  (id=1)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

✅ Model loaded on cuda


🚀 RUN 1/1

📁 teacher
   🔎 Searching: What is a course of study called? teacher...


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

   🤖 Answer: The context is irrelevant to the question....
   ✅ Saved 268/268 (Run 1, teacher)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🎉 Sperimentazione completata! I risultati sono in: risultati_web_agent.csv



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag